## Imports and config

In [5]:
import os
from azure.cosmos import CosmosClient
from openai import OpenAI 
from sentence_transformers import SentenceTransformer

# -- Cosmos DB Config --
COSMOS_URI = "https://ragnonprod-cosmos.documents.azure.com:443"
COSMOS_KEY = "UxK4Tqtbc3QKBtXR0YkyLza8fpsErQiCvhGHErrxPCoMl1dLpmHQc4wRY16X0Q8xPjN3K3DXqEXCACDb3kCKcA=="
DATABASE_NAME = "policy_rag_db"
CONTAINER_NAME = "EnronEmailVectorStore"

# -- Azure AI Foundry Config --
AZURE_AI_BASE_URL = "https://ragnonprod-ai.services.ai.azure.com/openai/v1" 
AZURE_AI_KEY = "7jDmXEi58VAH1n0LtBGVbZydJehSXsiMDY7HApKenfA6BOPfLAQsJQQJ99CFAC5RqLJXJ3w3AAAAACOG08OU"
CHAT_DEPLOYMENT = "gpt-4o-mini"

## Initialise Clients

In [6]:
# Initialize Cosmos DB Client
cosmos_client = CosmosClient(url=COSMOS_URI, credential=COSMOS_KEY)
database = cosmos_client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

# Initialize Client using the standard OpenAI class for Foundry
llm_client = OpenAI(
    base_url=AZURE_AI_BASE_URL,
    api_key=AZURE_AI_KEY
)

# 3. Initialize Local Embedding Model
print("Downloading/Loading local embedding model. This might take a moment on the first run...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Clients initialized successfully for Foundry!")

Downloading/Loading local embedding model. This might take a moment on the first run...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6070.93it/s]


Clients initialized successfully for Foundry!


## Core RAG Functions

In [ ]:
def get_embedding(text):
    """Generate vector embedding for input text.
    
    Converts the user's question into a vector using the local SentenceTransformer
    model. The embedding is serialized to a list format for Cosmos DB compatibility.
    
    Args:
        text (str): The input text to convert into an embedding vector.
    
    Returns:
        list: A list representation of the embedding vector.
    """
    # .encode() generates the embedding, .tolist() makes it JSON serializable for Cosmos
    return embedding_model.encode(text).tolist()

def vector_search(query_vector, top_k=3, sender_filter=None):
    """Search Cosmos DB for emails similar to query vector.
    
    Performs a vector similarity search on the Cosmos DB container to retrieve
    the most relevant emails. Search is faster when a sender_filter (partition key)
    is provided, avoiding cross-partition queries.
    
    Args:
        query_vector (list): The query embedding vector to search with.
        top_k (int, optional): Maximum number of results to return. Defaults to 3.
        sender_filter (str, optional): Email sender to filter by (partition key).
            If provided, restricts search to emails from this sender. Defaults to None.
    
    Returns:
        list: A list of email documents matching the search, sorted by similarity.
              Each email contains: subject, from, to, date, body, and similarity_score.
    
    Raises:
        Exception: If the Cosmos DB query fails. The exception is logged with
                   query and parameter details for debugging.
    """
    vector_str = str(query_vector)
    
    query = f"""
        SELECT TOP {top_k} c.subject, c["from"], c.to, c.date, c.body, 
        VectorDistance(c.vector, {vector_str}) AS similarity_score
        FROM c
    """
    
    parameters = []
    
    if sender_filter:
        query += "WHERE c.from = @sender "
        parameters.append({"name": "@sender", "value": sender_filter})
    
    query += f"ORDER BY VectorDistance(c.vector, {vector_str})"
   
    # Execute query
    try:
        if sender_filter:
            results = list(container.query_items(
                query=query, 
                parameters=parameters
            ))
        else:
            results = list(container.query_items(
                query=query, 
                parameters=parameters, 
                enable_cross_partition_query=True
            ))
        
        print(f"Found {len(results)} relevant emails in Cosmos DB.")
        return results
    except Exception as e:
        print(f"Query error: {e}")
        raise

def chat_with_enron_data(user_query):
    """Execute the RAG (Retrieval-Augmented Generation) pipeline.
    
    Orchestrates the complete RAG workflow: generates an embedding for the query,
    retrieves relevant emails from Cosmos DB, constructs context, and uses the
    LLM to generate a response based on retrieved context.
    
    Args:
        user_query (str): The user's question or query to answer.
    
    Returns:
        str: The assistant's response generated by the LLM based on retrieved
             context. Returns a message if no relevant emails are found.
    
    Raises:
        Exception: Propagates exceptions from embedding generation, search,
                   or LLM API calls.
    """
    print("1. Generating embedding for your query locally...")
    query_vector = get_embedding(user_query)
    
    print("2. Searching Cosmos DB for relevant emails...")
    retrieved_emails = vector_search(query_vector, top_k=3)
    
    if not retrieved_emails:
        return "I couldn't find any relevant emails in the database."
    
    context = ""
    for idx, email in enumerate(retrieved_emails):
        context += f"\n--- Email {idx + 1} ---\n"
        context += f"From: {email.get('from')}\n"
        context += f"To: {email.get('to')}\n"
        context += f"Date: {email.get('date')}\n"
        context += f"Subject: {email.get('subject')}\n"
        context += f"Body: {email.get('body')}\n"
    
    print("3. Generating response via Azure gpt-4o-mini...")
    
    system_prompt = (
        "You are a helpful AI assistant analyzing the Enron Email Dataset. "
        "Use ONLY the context provided below to answer the user's question. "
        "If the answer cannot be found in the context, clearly state that you do not know. "
        "Be concise and professional."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_query}"}
    ]
    
    response = llm_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=messages,
        temperature=0.3
    )
    
    return response.choices[0].message.content


## Interactive Conversation Loop

In [8]:
print("Welcome to the Enron Data Assistant. Type 'exit' or 'quit' to stop.")
print("-" * 50)

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Ending conversation. Goodbye!")
        break
        
    if not user_input.strip():
        continue
        
    try:
        answer = chat_with_enron_data(user_input)
        print(f"\nAssistant:\n{answer}")
        print("-" * 50)
    except Exception as e:
        print(f"\nAn error occurred: {e}")

Welcome to the Enron Data Assistant. Type 'exit' or 'quit' to stop.
--------------------------------------------------
1. Generating embedding for your query locally...
2. Searching Cosmos DB for relevant emails...
Found 3 relevant emails in Cosmos DB.
3. Generating response via Azure gpt-4o-mini...

Assistant:
The individuals who received mid-year promotions are:

1. Clapper, Karen - Sr Cust Svc Rep
2. Greaney, Chris - Sr Cust Svc Rep
3. Wilkens, Jerry - Sr Cust Svc Rep
4. Minton, Kevin - Pipeline Controller
5. Cox, Don - Pipeline Controller
6. Hanagriff, Richard - Sr Accounting Control Spec
--------------------------------------------------
Ending conversation. Goodbye!
